# Select sample size for temperature BMR

Selecting the number of samples is a balance between computational cost, given the high spatial resolution and associated memory requirements, and the need for a sufficiently large sample size to robustly span the uncertainty space.

The example we use here is for 1000 samples.

In [1]:
import os
import json
import glob
import pandas as pd
import xarray as xr
import numpy as np
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === CHOOSE NUMBER OF SAMPLES ===
n_samples = 1000

# Set the seed to produce reproducible random numbers
np.random.seed(42)

In [3]:
# === Set GBD version ===
GBD_version = "GBD23"

In [4]:
# === Health variables ===
health_vars = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
               "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
               "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
               "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
               "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
               "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

In [5]:
# === Path config ===
BMR_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR")
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")
ID_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23")

# Save file in scratch directory ~25GB
SAVE_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR")

In [6]:
# Load look up file
lookup_file = "country_location_ids.json"
lookup_path = os.path.join(ID_DIR, lookup_file)
with open(lookup_path, "r", encoding="utf-8") as f:
    country_to_id = json.load(f)

# Load country id mask
mask_file = "country_id_mask_0.25.nc"
mask_path = os.path.join(MASK_DIR, mask_file)
masks = xr.open_dataset(mask_path)

In [8]:
def project_to_global_map(bmr_da):
    bmr = bmr_da.transpose("country", "samples")

    # --- country -> id mapping ---
    countries = [str(c) for c in bmr["country"].values]
    ids_series = pd.Series(countries).map(country_to_id)
    unmatched = [c for c, i in zip(countries, ids_series) if pd.isna(i)]
    assert not unmatched, f"BMR countries with no ID in the JSON: {unmatched}"
    country_ids = ids_series.astype(int).values

    assert len(set(countries)) == len(countries), f"Duplicate country names in {bmr_file}"
    assert len(set(country_ids)) == len(country_ids), "Two BMR countries map to the same ID"
    vals = bmr.values  # (country, sample)
    assert np.isfinite(vals).all(), "NaN/inf BMR values present"

    unused_json = sorted(set(country_to_id) - set(countries))
    print(f"{len(countries)} countries x {n_samples} samples, all matched. JSON entries without BMR: {unused_json}")

    # --- mask checks ---
    ids = masks["country_id"].values  # (lat, lon)
    valid = np.isfinite(ids)
    ids_int = np.where(valid, ids, -1).astype(np.int64)
    assert np.all(ids[valid] == np.round(ids[valid])), "Non-integer IDs in mask"
    mask_ids = set(np.unique(ids_int[valid]).tolist())
    unknown = mask_ids - set(country_to_id.values())
    assert not unknown, f"Mask contains IDs not in the JSON: {sorted(unknown)}"

    # --- dense lookup table: (id, sample) ---
    lut = np.full((max(country_to_id.values()) + 1, n_samples), np.nan, dtype=np.float32)
    lut[country_ids, :] = vals

    # --- project onto grid: (sample, lat, lon) ---
    bmr_grid = np.full((n_samples, *ids.shape), np.nan, dtype=np.float32)
    bmr_grid[:, valid] = lut[ids_int[valid]].T   # (n_valid, sample).T -> (sample, n_valid)

    id_da = masks["country_id"]
    bmr_da = xr.DataArray(
        bmr_grid,
        coords={"samples": bmr["samples"].values, **id_da.coords},
        dims=("samples", *id_da.dims),
        name="BMR",
        attrs={**bmr.attrs,
               "grid_note": "Value of the country that covers the largest share of each 0.25 deg cell; NaN where no country.",
               "id_source": "country_location_ids.json"},
    )

    out = xr.Dataset({
        "BMR": bmr_da,
        "country_id": masks["country_id"],
        "country_id_frac": masks["country_id_frac"],
    })
    return out

In [9]:
# === Calculate the BMR distribution ===
for health_VAR in health_vars:
    print(f"Processing {health_VAR}")

    # Load BMR for each country (country, quantile)
    bmr_file = f"{GBD_version}_BMR_Country_{health_VAR}_newlabels_1990-2009.nc"
    bmr_path = os.path.join(BMR_DIR, bmr_file)
    BMR = xr.open_dataarray(bmr_path)  # three quantiles

    # BMR from VizHub (normal distribution)
    bmr_mean = BMR.sel(quantile="mean")
    bmr_lower = BMR.sel(quantile="lower")
    bmr_upper = BMR.sel(quantile="upper")
    bmr_std = (bmr_upper - bmr_lower) / (2 * 1.96)

    bmr_samples = np.random.normal(
        bmr_mean,
        bmr_std,
        size=(n_samples, len(BMR.country)))

    bmr_da = xr.DataArray(
        bmr_samples,
        dims=["samples", "country"],
        coords={"samples": np.arange(n_samples), "country": BMR.country}
    ).astype("float32")

    del bmr_samples

    bmr_global = project_to_global_map(bmr_da)

    # Save file ~25GB
    out_file = f"{GBD_version}_BMR_Global_Country_Map_era5_{health_VAR}_{n_samples}_samples_1990-2009.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    bmr_global.to_netcdf(out_path)

    del bmr_global

print("All processing complete.")

Processing CHRONIC_KIDNEY_DISEASE
204 countries x 1000 samples, all matched. JSON entries without BMR: []
Saving to /glade/derecho/scratch/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_CHRONIC_KIDNEY_DISEASE_1000_samples_1990-2009.nc
Processing CARDIOMYOPATHY_MYOCARDITIS
204 countries x 1000 samples, all matched. JSON entries without BMR: []
Saving to /glade/derecho/scratch/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_CARDIOMYOPATHY_MYOCARDITIS_1000_samples_1990-2009.nc
Processing HYPERTENSIVE_HEART_DISEASE
204 countries x 1000 samples, all matched. JSON entries without BMR: []
Saving to /glade/derecho/scratch/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_HYPERTENSIVE_HEART_DISEASE_1000_samples_1990-2009.nc
Processing ISCHEMIC_HEART_DISEASE
204 countries x 1000 samples, all matched. JSON entries without BMR: []
Saving to /glade/derecho/scratch/awells/workflow/BMR/GBD23_BMR_Global_Country_Map_era5_ISCHEMIC_HEART_DISEASE_1000_samples_1990-2009.nc
Processing STROK